# Influenza infection & immune response (Sego 2022 reproduction)

_Investigation `influenza-sego2022` — coder reproduction notebook._

**Question.** Can viva-cpm's native Cellular Potts engine reproduce, to quantitative figure
match, the cellularized multiscale influenza-infection and immune-response
model of Sego et al. 2022 — built on CompuCell3D's ViralInfectionVTM — at
paper scale (Figs 3B, 5, 7)?

A ground-up native reproduction of a published cellularized multiscale
infection model (Sego, Mochan, Ermentrout & Glazier 2022, J. Theor. Biol.
532:110918), used as a rigor exercise for viva-cpm's CPM engine: can it
reach quantitative figure match against a real, independently-published
spatial immunology model, using the authors' own CompuCell3D source as
ground truth rather than just the paper's prose? The investigation is
structured as a 10-increment capability ladder (Increment 0 spec/targets,
1-8 individual mechanisms at reduced scale, 9 the full capstone
reproduction) so reproducibility accrues cumulatively and each step is
independently reviewable.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-cpm/viva-cpm').is_dir():
    REPO = Path('/home/runner/work/viva-cpm/viva-cpm')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from pbg_cpm_studies.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Parameter provenance & digitized Fig 3B/5/7 targets (`parameter-provenance`)

**Question.** What is the authoritative, cited parameter set for reproducing Sego et al.
2022's cellularized influenza model (CompuCell3D ViralInfectionVTM), and
where does the source disagree with the paper's own printed tables?

**Objective.** Produce a single, cited parameter file and supporting provenance
documentation that later increments can consume without re-deriving
constants from the paper or source, and digitize the paper's Figs 3B, 5, 7
into acceptance-band targets for the eventual capstone reproduction.

**Hypothesis.** N/A — this is a documentation/provenance study, not a hypothesis-testing
simulation. No hypothesis is evaluated here.

**Purpose.** Documentation only — no simulation is run by this study. It establishes
the parameter/spec authority that all later increments build on.

**Claim.** The parameter authority for the influenza-sego2022 investigation is
established and cited: params.yaml is sourced from the CC3D
ViralInfectionVTM package and the paper's Tables 1-4, with every value
traceable to one or the other, and figure-target acceptance bands exist for
Figs 3B, 5, 7. This is a claim of DOCUMENTATION COMPLETENESS, not of
reproduction — no simulation has been run against these targets.


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: parameter-provenance ===
STUDY = 'parameter-provenance'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Acceptance-band targets (Fig 3B)**


In [ ]:
# Acceptance-band targets (Fig 3B)
show_viz(_render_one('local:InfluenzaFig3BTargets', {}, RUNS_DB, STUDY_YAML))

## Study: Epithelial sheet baseline (`epithelial-sheet-baseline`)

**Question.** Does a confluent tiling of epithelial (type H) cells, using the CC3D
source-literal adhesion values (H-H=5.0, H-medium=25.0) and target volume
(25 sites), hold stable under CPM relaxation at paper scale, and does the
Rust engine step it fast enough for the eventual 2-week/~20160-MCS
capstone run?

**Objective.** Build the epithelial-sheet `load_world` spec (Task 1.1), instantiate and
relax the CPM world (Task 1.2), and measure step throughput on the full
1mm^2 patch (Task 1.3). Wrap the geometry in a process-bigraph composite
(`pbg_cpm_studies.composites.influenza.epithelium`) so the
dashboard can run a modest (0.1mm, 100-cell) live demo of the same
substrate (Task 1.4, this study).

**Hypothesis.** The confluent sheet holds: after relaxation every real cell (indices 1..N,
excluding the medium placeholder at index 0) has volume close to its
target of 25 sites, with none vanishing. Separately, 1mm^2 throughput
comfortably clears the ~2.0 MCS/s floor required for the capstone's
~20160-step run to finish in a practical wall-clock time.

**Purpose.** epithelial_sheet_substrate

**Claim.** The confluent epithelial-only sheet is geometrically stable (mean cell
volume ~25 sites, none vanished) and the Rust CPM engine steps the full
1mm^2 lattice at ~406-422 MCS/s, far above the throughput needed to make
the eventual 2-week capstone run computationally tractable. This is a
substrate-readiness claim, not a biological-reproduction claim.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_cpm_studies.composites.influenza.epithelium` | 0 | patch_mm=0.1 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.influenza.epithelium`** — `spec_pbg_cpm_studies_composites_influenza_epithelium` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.influenza.epithelium` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: epithelial-sheet-baseline ===
STUDY = 'epithelial-sheet-baseline'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Spatial state — sheet relaxation (animated)**


In [ ]:
# Spatial state — sheet relaxation (animated)
show_viz(_render_one('local:InfluenzaSpatialSheet', {}, RUNS_DB, STUDY_YAML))

**Confluent epithelial sheet**


In [ ]:
# Confluent epithelial sheet
show_viz(_render_one('local:InfluenzaEpithelialSheet', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| Sheet holds confluent after relaxation | kind=mean_cell_volume_sites condition=epithelial-sheet-baseline stat=mean | op gt value 0 |
| 1mm^2 throughput clears the perf-gate floor | kind=throughput_mcs_per_s condition=epithelial-sheet-baseline stat=min | op gt value 200.0 |


## Study: Virus field & stochastic H→I infection (`virus-field-infection`)

**Question.** Does the extracellular virus field (source-cited diffusion coefficient and
decay rate, secreted by InfectedReleasing cells) diffuse and decay as
specified, and does a stochastic, field-driven H -> I infection transition
spread a seeded lesion LOCALLY (new infections cluster near existing
infected cells) rather than uniformly, with total cell population
conserved?

**Objective.** Wire the virus field (Task 2.1, `pbg_cpm_studies/influenza/fields.py`) and
the stochastic infection transition (Task 2.2,
`pbg_cpm_studies/influenza/transitions.py`) together over the Increment-1
sheet, drive them for 60 updates from a small seeded lesion
(`pbg_cpm_studies/influenza/run.py::run_virus_infection`, Task 2.3), and
report the measured spread/locality/null-control behavior from the
integration test (`tests/test_influenza_virus_infection.py`). Wrap the
same mechanism as a process-bigraph composite
(`pbg_cpm_studies.composites.influenza.viral_infection`, CPMProcess +
InfectionProcess wired through `fates`) for the dashboard live demo (Task
2.4, this study) — measurements come from `run_virus_infection`, not the
live-demo composite (see caveat and Task 2.3's report on why the two
paths use independent RNG streams and are not bit-identical).

**Hypothesis.** Layered on the Increment-1 confluent sheet: a virus field secreted by a
small seeded lesion of InfectedReleasing (I) cells diffuses outward and
decays; a stochastic H -> I transition (rate = g_hv * local mean field
concentration) converts nearby Healthy (H) cells to Infected (I) over
time, with new infections landing close to already-infected cells (within
a few multiples of the field's own diffusion length) rather than
scattered across the whole sheet, and n_H + n_I conserved every update.

**Purpose.** virus_field_and_infection_transition

**Claim.** The extracellular virus field diffuses and decays (source-cited
parameters), and a seeded lesion of InfectedReleasing cells spreads via a
stochastic, virus-driven H -> I transition to nearby Healthy cells
(locally, within a few diffusion lengths), with total cell population
conserved every update. This is a MECHANISM-validation claim (the field
and the transition behave sensibly together), not a claim that any of
Sego et al. 2022's quantitative figure targets (Figs 3B/5/7) are met —
that reproduction verdict remains PENDING until Increment 9 (the
capstone).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_cpm_studies.composites.influenza.viral_infection` | 0 | patch_mm=0.1, seed=17, init_infected_frac=0.05 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.influenza.viral_infection`** — `spec_pbg_cpm_studies_composites_influenza_viral_infection` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.influenza.viral_infection` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: virus-field-infection ===
STUDY = 'virus-field-infection'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Spatial state — infection spread (animated)**


In [ ]:
# Spatial state — infection spread (animated)
show_viz(_render_one('local:InfluenzaSpatialVirusField', {}, RUNS_DB, STUDY_YAML))

**Virus field + local spread (animated)**


In [ ]:
# Virus field + local spread (animated)
show_viz(_render_one('local:InfluenzaVirusFieldScene', {}, RUNS_DB, STUDY_YAML))

**Infection dynamics + locality**


In [ ]:
# Infection dynamics + locality
show_viz(_render_one('local:InfluenzaInfectionDynamics', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| Seeded lesion spreads over time (population conserved) | kind=n_I condition=virus-field-infection stat=delta | op gt value 0 |
| New infections land closer to prior infection than random chance (locality) | kind=locality_null_ratio condition=virus-field-infection stat=mean | op lt value 0.75 |
| No seed, no initial virus -> no infection (null control) | kind=n_I condition=virus-field-infection-null stat=max | op eq value 0 |


## Study: Type-I IFN field & cellular resistance (`ifn-resistance`)

**Question.** Does a diffusing type-I IFN field, converted per-cell into a resistance
scalar via the CC3D-source formula `resist = f_bar/(a_rf+f_bar)`, gate
infected cells' virus secretion by `(1-resist)` strongly enough to
measurably slow infection spread and reduce total virus versus the same
seed/parameters with no IFN (Increment 2's unmodified path)?

**Objective.** Wire the type-I IFN field (Task 3.2, `pbg_cpm_studies/influenza/fields.py`
`add_ifn_field`) and the pure per-cell resistance function (Task 3.3,
`pbg_cpm_studies/influenza/resistance.py::cell_resistance`) into a new
driver, `run.run_virus_infection_with_ifn` (Task 3.3), alongside the
unmodified Increment-2 `run_virus_infection`, and compare their per-update
series (n_I, n_H, total_virus, plus `mean_resist` recorded only by the
with-IFN driver) at identical seed/parameters (Task 3.3's integration test
`tests/test_influenza_virus_infection.py::test_ifn_resistance_slows_spread_vs_no_ifn`
and the fuller with/without series in Task 3.3's report). A minimal
in-package figure (`pbg_cpm_studies/influenza/viz.py::ifn_resistance_figure`,
Task 3.4) renders both series side by side plus the mean_resist trajectory.

**Hypothesis.** Layered on the Increment-2 virus field + infection transition: a type-I
IFN field secreted by infected cells diffuses and decays; each infected
cell's locally-sampled mean IFN drives a per-cell resistance scalar that
throttles ITS OWN virus secretion for the next update. This should leave
n_I lower and total_virus substantially lower than an identical no-IFN
run at the same steps, while mean_resist stays strictly positive once IFN
has had time to accumulate.

**Purpose.** ifn_field_and_per_cell_resistance

**Claim.** A type-I IFN field, sampled locally per infected cell into a resistance
scalar (source formula) that gates that cell's virus secretion by
`(1-resist)`, measurably reduces total virus versus an identical no-IFN
run (same seed, same 60-update driver) — roughly HALVED at every sampled
step (819.16 vs 1750.69 at step 20, ~53%; 1222.88 vs 2752.34 at step 30,
~56%; 1624.58 vs 3905.38 at step 40, ~58%). This is the study's PRIMARY,
robust quantitative evidence. Infected-cell count (n_I) moves in the same
protective direction but is a WEAKER signal at the step Task 3.3's
integration test actually asserts: 48 vs 50 at step 20 is only a ~4%
reduction; the gap widens to ~15% at step 30 (51 vs 60) and ~23% at step
40 (53 vs 69) as the virus-load reduction has more time to compound into
a visible count difference. So n_I is treated here as a corroborating
directional trend, not co-equal quantitative evidence with total_virus.
This is a MECHANISM-validation claim (the IFN->resistance->reduced-
secretion chain behaves in the protective direction and has a real,
non-trivial magnitude on virus load, with a smaller/slower-to-emerge
effect on cell counts), not a claim that any of Sego et al. 2022's
quantitative figure targets (Figs 3B/5/7) are met — that reproduction
verdict remains PENDING until Increment 9 (the capstone).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_cpm_studies.composites.influenza.viral_infection` | 0 | patch_mm=0.1, seed=17, init_infected_frac=0.05 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.influenza.viral_infection`** — `spec_pbg_cpm_studies_composites_influenza_viral_infection` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.influenza.viral_infection` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: ifn-resistance ===
STUDY = 'ifn-resistance'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Spatial state — IFN-gated infection (animated)**


In [ ]:
# Spatial state — IFN-gated infection (animated)
show_viz(_render_one('local:InfluenzaSpatialIfnResistance', {}, RUNS_DB, STUDY_YAML))

**IFN + per-cell resistance**


In [ ]:
# IFN + per-cell resistance
show_viz(_render_one('local:InfluenzaIfnResistance', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| IFN-gated resistance does not let spread outrun the no-IFN path (weak directional check) | kind=n_I_delta_vs_without_ifn condition=ifn-resistance stat=final | op le value 0 |
| IFN-gated virus release leaves strictly less total virus (primary quantitative gate) | kind=total_virus_delta_vs_without_ifn condition=ifn-resistance stat=final | op lt value 0 |
| Resistance gate actually engages (mean_resist > 0) | kind=mean_resist condition=ifn-resistance stat=final | op gt value 0.0 |


## Study: Epithelial death & Allee recovery (`epithelial-fate`)

**Question.** Does wiring infection (H->I), infected death (I->D), and the cellularized
Allee effect (H->D death / D->H recovery, both gated by local contact
geometry) together produce a coherent epithelial-fate lifecycle -- a
growing dead lesion, population conservation, and correct bidirectional
Allee response (recovery for well-surrounded D cells, no recovery for
poorly-surrounded ones) -- on top of the Increment 2/3 virus/infection/IFN
path?

**Objective.** Wire infected death (Task 4.2, `transitions.infected_death_step`) and the
cellularized Allee effect (Task 4.3, `pbg_cpm_studies/influenza/allee.py`)
into a new driver, `run.run_epithelial_fate` (Task 4.3), alongside the
unmodified Increment-2/3 infection + IFN-resistance path, and measure the
resulting n_H/n_I/n_D series plus the diagnostic n_allee_death/
n_allee_recovery counters (Task 4.3's integration tests
`tests/test_influenza_allee.py::test_lesion_forms_dead_region_grows_over_time`,
`::test_dead_cell_surrounded_by_uninfected_recovers`,
`::test_dead_cell_surrounded_by_dying_does_not_recover`, and the fuller
200/500-step series in task-4.3-report.md). A minimal in-package figure
(`pbg_cpm_studies/influenza/viz.py::epithelial_fate_figure`, Task 4.4)
renders the cell-type composition over time plus cumulative Allee event
counts.

**Hypothesis.** Layered on the Increment-2/3 virus field + infection + IFN-resistance
path: infected cells die at rate `mu_i*(1-resist)` (I->D), and every H/D
epithelial cell's local contact-surface composition (restricted to H/I/D
neighbors) drives a stochastic Allee death (H->D, for H cells deeply
embedded in dead tissue) or recovery (D->H, for D cells mostly surrounded
by healthy tissue) rate. This should produce a growing, population-
conserving dead lesion over time, with D cells embedded in healthy tissue
recovering and D cells embedded in dying tissue never recovering -- while
direct Allee death may be rare at reduced (0.3mm) scale given the
source's b_h/theta asymmetry.

**Purpose.** epithelial_fate_lifecycle

**Claim.** Wiring infection, infected death, and the cellularized Allee effect
together produces a coherent, population-conserving epithelial-fate
lifecycle: a dead lesion grows via H->I->D (n_D: 0->14 at 200 updates,
0->77 at 500 updates, same seed=17/0.3mm/900-cell sheet as Increments
2/3), and the Allee branch responds correctly to local contact geometry
in BOTH directions -- a dead cell fully surrounded by healthy tissue
recovers (proven both deterministically in a hand-made frozen-geometry
world and organically in the full driver, 3-12 events/run), while a dead
cell surrounded by dying tissue never does (rate is provably 0, not just
empirically absent). This is a MECHANISM-validation claim (the fate
lifecycle composes correctly and the Allee branch is bidirectionally
functional), not a claim that any of Sego et al. 2022's quantitative
figure targets (Figs 3B/5/7) are met -- that reproduction verdict remains
PENDING until Increment 9 (the capstone). Direct H->D Allee death is
honestly reported as RARE at this reduced scale (0 observed events across
both runs) -- verified genuine via rate instrumentation (15,447
qualifying encounters, all correctly nonzero, expected ~0.19 successes),
not a wiring bug, and flagged as an open Increment-9 calibration question
rather than tuned away here.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_cpm_studies.composites.influenza.viral_infection` | 0 | patch_mm=0.1, seed=17, init_infected_frac=0.05 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.influenza.viral_infection`** — `spec_pbg_cpm_studies_composites_influenza_viral_infection` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.influenza.viral_infection` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: epithelial-fate ===
STUDY = 'epithelial-fate'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Spatial state — fate lifecycle (animated)**


In [ ]:
# Spatial state — fate lifecycle (animated)
show_viz(_render_one('local:InfluenzaSpatialEpithelialFate', {}, RUNS_DB, STUDY_YAML))

**Epithelial-fate lifecycle**


In [ ]:
# Epithelial-fate lifecycle
show_viz(_render_one('local:InfluenzaEpithelialFate', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| Epithelial-fate lifecycle forms a growing dead lesion, population conserved | kind=n_D_final condition=epithelial-fate stat=final | op gt value 0 |
| Dead cell fully surrounded by healthy tissue recovers (D->H) | kind=recovers_within_500_draws condition=epithelial-fate stat=final | op eq value True |
| Dead cell surrounded by dying tissue never recovers (negative control) | kind=recovery_rate_over_500_draws condition=epithelial-fate stat=max | op eq value 0.0 |


## Study: Macrophage localization to infection (`macrophage-response`)

**Question.** Does wiring the macrophage cell type to chemotax up the extracellular-virus
gradient (via the engine's existing `World.set_chemotaxis` primitive, no
Rust change) produce genuine spatial LOCALIZATION to the infection -- mean
macrophage-to-infection distance decreasing over time -- robustly across
multiple seeds, and with a geometrically UNBIASED (interior-placed, not
wall-pinned) lambda=0 control that shows no such localization?

**Objective.** Wire the macrophage cell type (`types.M`, Task 5.0) and virus-field
chemotaxis (`immune.set_macrophage_chemotaxis`, Task 5.1) into a
non-confluent 2D-approximation scenario
(`immune.build_macrophage_scenario_spec`, fix round 1: both clusters
interior-placed) and driver (`run.run_macrophage_response`, fix round 1:
fixed non-tuned knobs), and measure the resulting
`mean_distance_to_infection` series with chemotaxis on (lambda=5000,
`params.yaml` default) vs off (lambda=0 control), across 5 seeds (5, 17,
23, 42, 100) (Task 5.1's integration test
`tests/test_influenza_macrophage.py::test_macrophages_localize_to_infection_across_seeds`,
which asserts ON is negative in every seed, ON < OFF in every seed, and
the mean on/off gap exceeds OFF's own seed-to-seed spread; plus the
minimal 2-cell engine-sanity tests, and the fuller localization-number
table recorded in task-5.1-report.md's fix-round-1 section). A minimal
in-package figure (`pbg_cpm_studies/influenza/viz.py::macrophage_response_figure`,
Task 5.2) renders the mean-distance-to-infection trajectory (on vs off
overlay) plus the macrophage centre-of-mass path for a single representative
run.

**Hypothesis.** Macrophages (`types.M`) placed in the domain interior, away from a seeded
infection also placed in the interior (neither cluster wall-pinned), with
chemotaxis wired toward the (unchanged, Increment-2) virus field via the
engine's linear-lambda `World.set_chemotaxis` primitive, will move up the
virus gradient and accumulate near the infection over time -- their mean
centre-of-mass distance to the infected-cell centroid should decrease
substantially and consistently across multiple seeds. A lambda=0 control
(chemotaxis mechanically disabled, same seeds/scenario otherwise) should
show a small, directionless change -- isolating the chemotaxis mechanism
(rather than incidental wall drift or adhesion) as the cause of
localization, and ruling out a seed-specific curve-fit.

**Purpose.** macrophage_chemotactic_localization

**Claim.** Wiring macrophage chemotaxis to the (unchanged) virus field via the
engine's existing linear-lambda primitive produces genuine spatial
LOCALIZATION, robust across seeds and against a geometrically unbiased
control: with both the macrophage cluster and the infection placed in the
domain interior (start distance 47.20 sites, identical across seeds),
chemotaxis-on reduces the macrophages' mean distance to the infection
centroid in EVERY one of 5 tried seeds (mean change -13.39 sites, range
-11.52 to -17.75), while the SAME starting configuration with chemotaxis
off (lambda=0 control) shows a small, directionless change (mean -0.85,
range -4.99 to +3.45) -- near-isotropic, not a masked drift. Chemotaxis-on
beats the lambda=0 control in every individual seed (paired comparison),
and the mean on/off gap (-12.54) exceeds the control's own seed-to-seed
spread (8.44), so the effect is not explainable by control noise alone.
This is a LOCALIZATION-MECHANISM validation claim (macrophages chemotax
toward the infection, matching Fig 2B/3A's qualitative direction, and now
demonstrated to replicate rather than asserted from a single seed), NOT a
claim that any quantitative reproduction target is met, that macrophages
phagocytose virus (the CC3D source has no such per-macrophage mechanism),
or that recruitment dynamics are modeled (a fixed population is placed at
scenario build time) -- that fuller reproduction verdict remains PENDING
until Increment 9 (the capstone). The 5-seed ensemble is a modest sample,
not a full statistical power analysis or source-scale replicate count
(see caveats).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_cpm_studies.composites.influenza.innate_immunity` | 0 | chemotaxis_v_macro=5000.0, seed=17 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.influenza.innate_immunity`** — `spec_pbg_cpm_studies_composites_influenza_innate_immunity` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.influenza.innate_immunity` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: macrophage-response ===
STUDY = 'macrophage-response'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Spatial state — macrophage recruitment (animated)**


In [ ]:
# Spatial state — macrophage recruitment (animated)
show_viz(_render_one('local:InfluenzaSpatialMacrophage', {}, RUNS_DB, STUDY_YAML))

**Macrophage localization**


In [ ]:
# Macrophage localization
show_viz(_render_one('local:InfluenzaMacrophageResponse', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| Chemotaxis-on reduces distance to infection in every tried seed | kind=all_seeds_on_distance_change_negative condition=macrophage-response-on-multiseed stat=final | op eq value True |
| Chemotaxis-on ends closer than the lambda=0 control in every seed (paired comparison) | kind=all_seeds_on_lt_off condition=macrophage-response-on-vs-off-multiseed stat=final | op eq value True |
| Mean on/off gap exceeds the control's own seed-to-seed spread | kind=mean_gap_exceeds_control_spread condition=macrophage-response-gap-vs-spread stat=final | op gt value control_spread |
| Engine-sanity — macrophage moves up the virus gradient | kind=distance_final_lt_initial condition=engine-sanity-on stat=final | op lt value distance[0] |
| Engine-sanity — no approach without chemotaxis (negative control) | kind=distance_final_ge_initial condition=engine-sanity-off stat=final | op ge value distance[0] |


## Study: Chemokine & IL-10 signaling fields (`signaling-fields`)

**Question.** Do the macrophage-released chemokine and IL-10 diffusible fields (Task
6.1) actually form the expected spatial and regulatory structure: a
chemokine GRADIENT centered on the macrophage cluster (higher near, lower
far, decaying with distance -- the attractant profile future NK/CD8+
recruitment will chemotax up), and IL-10 positive/increasing from BOTH its
regulated sources (macrophage Hill self-regulation, uninfected 1-resist
gating)?

**Objective.** Wire the chemokine + IL-10 fields (`fields.add_chemokine_field`/
`add_il10_field`, Task 6.1) and their per-cell secretion regulation
(`signaling.macrophage_secretion_scale`/`uninfected_il10_scale`) into the
Increment-5 non-confluent macrophage scenario, and drive
`run.run_macrophage_signaling` (steps=15, seed=17, default scenario
knobs) to measure: (a) the chemokine field's per-cell near (macrophages)
vs. far (uninfected epithelial patch, `separation_sites` away) split, and
its raw-lattice radial profile (6 concentric bins) around the macrophage-
cluster centroid; (b) IL-10 positivity and growth at both the macrophage
and uninfected-cell source populations. Evidence: `tests/
test_influenza_signaling.py`'s 4 integration tests
(`test_chemokine_and_il10_fields_are_positive_after_macrophage_signaling_run`,
`test_chemokine_field_is_highest_near_macrophages_and_decays_with_distance`,
`test_il10_positive_from_macrophage_and_uninfected_sources`,
`test_macrophage_signaling_run_is_deterministic`) plus the full
localization numbers recorded in task-6.1-report.md. A minimal in-package
figure (`pbg_cpm_studies/influenza/viz.py::signaling_fields_figure`, Task
6.2) renders the chemokine radial profile (panel a) and the IL-10
trajectories at both source populations (panel b).

**Hypothesis.** Wiring macrophage-released chemokine secretion (Hill-self-regulated by
each macrophage's own local IL-10) and dual-source IL-10 secretion
(macrophage Hill-regulated + uninfected 1-resist-gated) as diffusible
fields over the Increment-5 macrophage scenario will produce a chemokine
concentration that is substantially higher at/near the macrophage cluster
than far from it, decaying with distance in a position-independent radial
profile -- and both IL-10 sources will register positive, increasing
concentration, neither pinned at exactly 0 (the secretion never engages)
nor saturating instantly (the Hill/resist gates are degenerate).

**Purpose.** chemokine_il10_field_gradient

**Claim.** Wiring macrophage-released chemokine + IL-10 diffusible fields, with
per-cell secretion regulated by a shared IL-10-Hill self-regulation loop
(macrophages) and a `1-resist` gate (uninfected epithelial cells),
produces a genuine chemokine GRADIENT centered on the macrophage cluster:
per-cell, macrophages read 0.00261 vs. 0.00062 at a far uninfected
epithelial patch (4.2x near/far), and a raw-lattice radial profile
(position-independent) decays monotonically from 0.00195 at the
macrophage-centered innermost bin to 0.00033 at the outermost of 6 bins
(5.9x end-to-end drop). IL-10 is positive and increasing from BOTH its
regulated sources (macrophages 2.97e-4, uninfected cells 3.71e-5, final
step). This is a FIELD-MECHANISM validation claim (the gradient future
NK/CD8+ recruitment, Increment 7, will chemotax up genuinely exists and is
correctly centered) -- NOT a claim that any quantitative reproduction
target is met, that `sig_1`'s true TNF-dependent dynamics are modeled (a
documented constant stub is used, Increment 8), or that recruitment/
chemotaxis toward the field is wired (Increment 7). That fuller
reproduction verdict remains PENDING until Increment 9 (the capstone).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_cpm_studies.composites.influenza.innate_immunity` | 0 | chemotaxis_v_macro=5000.0, seed=17 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.influenza.innate_immunity`** — `spec_pbg_cpm_studies_composites_influenza_innate_immunity` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.influenza.innate_immunity` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: signaling-fields ===
STUDY = 'signaling-fields'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Spatial state — signaling scene (animated)**


In [ ]:
# Spatial state — signaling scene (animated)
show_viz(_render_one('local:InfluenzaSpatialSignaling', {}, RUNS_DB, STUDY_YAML))

**Chemokine + IL-10 fields**


In [ ]:
# Chemokine + IL-10 fields
show_viz(_render_one('local:InfluenzaSignalingFields', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| Chemokine and IL-10 fields are positive after the run | kind=total_field_positive_and_increasing condition=signaling-fields-positivity stat=final | op eq value True |
| Chemokine is highest near macrophages and decays with distance | kind=chemo_near_gt_far_and_radial_decay condition=signaling-fields-gradient stat=final | op eq value True |
| IL-10 positive from both macrophage and uninfected sources | kind=il10_dual_source_positive condition=signaling-fields-il10-sources stat=final | op eq value True |
| Run is fully deterministic given a fixed seed | kind=series_equal_across_repeats condition=signaling-fields-determinism stat=all | op eq value True |


## Study: NK / CD8⁺ cytotoxic killing (`cytotoxic-killing`)

**Question.** Do NK (type K) and CD8+ T (type E) cells, chemotaxing up the macrophage-
released chemokine field and contact-killing infected cells via the CC3D
source's literal LOCAL surface-contact kill-rate formula, genuinely
LOCALIZE to the infection (robust across seeds) and genuinely CLEAR an
infected cell once contact happens -- and does that killing mechanism
actually engage end-to-end within a feasible step budget at the
scenario's own DEFAULT full scale, or only in an artificially
close-contact test?

**Objective.** Wire NK (`types.K`)/CD8+ (`types.E`) chemotaxis (`immune.
set_nk_cd8_chemotaxis`, Task 7.1) up the chemokine field (Increment 6)
and contact-killing (`killing.contact_kill_rate`, Task 7.2, the literal
`ContactKillingSteppable` LOCAL form) into the Task-7.1 three-cluster
scenario (`immune.build_cytotoxic_scenario_spec`) and driver (`run.
run_cytotoxic_response`), and measure: (a) NK/CD8 `mean_distance_to_
infection` across 5 seeds (5, 17, 23, 42, 100), chemotaxis on (lambda x
`NK_CD8_CHEMOTAXIS_ENGINE_SCALE=100`) vs a lambda=0 control (`tests/
test_influenza_nk_cd8.py::test_nk_and_cd8_localize_to_infection_across_seeds`,
task-7.1-report.md); (b) `n_infected` in a small close-contact scenario,
killing enabled vs disabled, same seed (`tests/test_influenza_killing.py::
test_infected_count_lower_with_nk_cd8_killing_than_without`,
task-7.2-report.md); (c) `n_infected` at the scenario's own DEFAULT full
scale with killing enabled, an ad hoc (not fast-pytest-suite) measurement
recorded honestly in task-7.2-report.md's "Honest assessment" section. A
minimal in-package figure (`pbg_cpm_studies/influenza/viz.py::
cytotoxic_killing_figure`, Task 7.3) renders the NK+CD8 distance
trajectory (on vs control overlay) plus the infected-count trajectory
(killing enabled vs disabled overlay).

**Hypothesis.** NK/CD8 cells placed in the domain interior, chemotaxing toward the
macrophage-released chemokine field via the engine's linear-lambda
`World.set_chemotaxis` primitive, will move up the gradient and
accumulate near the infection, robustly across seeds, with CD8 (2x NK's
literal lambda) localizing more strongly. Once NK/CD8 are in actual
lattice contact with an infected cell, `killing.contact_kill_rate`'s
stochastic draw will remove it (`types.I` -> `types.D`). At the
scenario's own default full scale, however, NK/CD8 may not close the full
distance to the infected cell within a feasible step budget -- if so,
that should be reported honestly as a weak end-to-end result, not
concealed by re-tuning the scenario/lambda until it "works."

**Purpose.** nk_cd8_chemotactic_localization_and_contact_killing

**Claim.** NK/CD8 chemotaxis up the (unchanged, Increment-6) chemokine field
produces genuine spatial LOCALIZATION, robust across 5 tried seeds (NK
mean distance change -20.37, CD8 -35.99, both negative in 5/5 seeds,
against a near-isotropic lambda=0 control, NK -1.02/CD8 -0.54) with CD8
(2x NK's lambda) localizing more strongly than NK in every seed and on
average -- matching the paper's Sec. 2.3 "CD8+ sensitivity twice that of
NK cells." Separately, the contact-killing MECHANISM itself
(`killing.contact_kill_rate`, the source's literal LOCAL surface-contact
form, resist-DIRECT per discrepancy #7) is proven correct: a small
close-contact scenario kills the one infected cell (1 -> 0 at update 22
of 45) with killing enabled vs steady at 1 with the identical seed and
killing disabled. THIS STUDY DOES NOT CLAIM "NK/CD8 CLEAR THE INFECTION"
AT REALISTIC SCALE -- at the scenario's own DEFAULT full scale (same
6-macrophage/6-NK/6-CD8, 60-update scenario the localization numbers
above come from), NK/CD8 close much of the distance to the infection
(NK ~81.6 -> ~57.8, CD8 ~81.6 -> ~39.8) but NEVER reach actual contact
within the run budget, so `n_infected` is unchanged (stays at 1 for all
61 recorded steps, no kill fires). The killing CAPABILITY is
demonstrated; end-to-end cytotoxic CLEARANCE at default/realistic scale
is an OPEN Increment-9 field-magnitude-calibration item, the same root
cause as the localization mechanism's own 100x engine-unit-scale
correction, deliberately NOT tuned away here.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_cpm_studies.composites.influenza.cytotoxic_immunity` | 0 | chemotaxis_v_macro=5000.0, chemotaxis_v_nk=500000.0, chemotaxis_v_cd8=1000000.0, seed=17 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.influenza.cytotoxic_immunity`** — `spec_pbg_cpm_studies_composites_influenza_cytotoxic_immunity` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.influenza.cytotoxic_immunity` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: cytotoxic-killing ===
STUDY = 'cytotoxic-killing'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Spatial state — cytotoxic response (animated)**


In [ ]:
# Spatial state — cytotoxic response (animated)
show_viz(_render_one('local:InfluenzaSpatialCytotoxic', {}, RUNS_DB, STUDY_YAML))

**NK/CD8 localization + killing**


In [ ]:
# NK/CD8 localization + killing
show_viz(_render_one('local:InfluenzaCytotoxicKilling', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| NK localization -- distance to infection decreases in every tried seed | kind=all_seeds_nk_on_distance_change_negative condition=cytotoxic-nk-on-multiseed stat=final | op eq value True |
| CD8 localization -- distance to infection decreases in every tried seed | kind=all_seeds_cd8_on_distance_change_negative condition=cytotoxic-cd8-on-multiseed stat=final | op eq value True |
| Contact-killing reduces infected count vs. killing-disabled control | kind=n_infected_final_with_killing_lt_without condition=close-contact-killing-on-vs-off stat=final | op lt value n_infected_without_killing[final] |
| Killing-disabled control holds an identical scenario | kind=scenario_params_match_across_killing_flag condition=killing-control-scenario-identity stat=final | op eq value True |
| contact_kill_rate matches params.yaml's precomputed coefficients | kind=contact_kill_rate_matches_precomputed_coefficients condition=contact-kill-rate-unit stat=final | op eq value True |


## Study: Global Price-2015 ODE coupling (`global-coupling`)

**Question.** Does coupling the Price (2015)-derived global immune ODE (10 systemic
species) bidirectionally to the spatial CPM patch — dynamic sig_1
feedback, ODE-driven recruitment, and a well-mixed nearby-population
killing term — resolve the three stubs carried since Increments 5-7 into
mechanisms that are individually correct and source-faithful, and do the
resulting coupling magnitudes already reconcile at this reduced-scale
patch, or is that an open Increment-9 calibration item?

**Objective.** Wire `price_ode.GlobalODE` (Task 8.1, the hybrid 10-species integrator,
discrepancy #9) into `run.run_global_coupling`'s per-MCS loop: spatial->
ODE aggregate push (Task 8.2), ODE->spatial dynamic sig_1 feedback (Task
8.3, discrepancy #11), ODE-driven recruitment (Task 8.4, discrepancy
#12), and NK/CD8 LOCAL+NEARBY cytotoxic killing (Task 8.5). Measure sig_1's
dynamic range vs the retired static stub, recruitment inflow/outflow
rates and population trajectories (default vs boosted eta), and whether
the nearby-killing term fires via natural recruitment buildup vs a
documented test-seam override
(`tests/test_influenza_price_ode.py`, `tests/test_influenza_recruitment.py`,
`tests/test_influenza_killing.py`, task-8.1 through task-8.5-report.md).
A minimal in-package figure (`pbg_cpm_studies/influenza/viz.py::
global_coupling_figure`, Task 8.6) renders the systemic ODE species
(T/X/A/P) and the spatial aggregates (I/M/K/E) plus dynamic sig_1 over
MCS.

**Hypothesis.** Integrating the 10 systemic ODE species once per MCS, fed by the spatial
model's cell counts/field integrals and feeding back a dynamic sig_1 plus
ODE-driven recruitment plus a nearby-population killing term, will
produce mechanisms that are each individually verifiable as correct
(dynamic sig_1 tracks TNF/dead count; recruitment shows the source's
documented CD8-no-baseline asymmetry; nearby-killing clears infected
cells given a sufficient nearby population) — but at this driver's small
reduced-scale default (tiny eta), the resulting coupling magnitudes may
not yet be large enough to produce an observable effect within a
feasible step budget, which should be reported honestly as a
calibration gap deferred to Increment 9, not concealed by tuning
constants to force a better-looking number.

**Purpose.** hybrid_price2015_global_ode_bidirectional_coupling

**Claim.** The hybrid Price-2015 global ODE (10 systemic species, discrepancy #9)
integrates stably once per MCS and responds to the spatial infection.
All three stubs carried since Increments 5-7 are RESOLVED as dynamic,
source-faithful mechanisms: dynamic sig_1 (`a_11*T + a_12*D`, the
unchanged Michaelis secretion form, discrepancy #11), ODE-driven
recruitment (chemokine/APC Hill inflows with the source's CD8-no-
baseline asymmetry preserved, discrepancy #12), and NK/CD8 killing
extended with a well-mixed NEARBY term (verified via a documented test
seam) alongside the unchanged Increment-7 LOCAL contact term. THIS STUDY
DOES NOT CLAIM THESE MECHANISMS ARE ALREADY CALIBRATED — at this
driver's default reduced-scale patch (eta ~1.4e-4), dynamic sig_1's
value is ~6 orders of magnitude below the retired static stub (2.44),
collapsing the secretion-scale factor to near-zero; ODE-driven
recruitment produces sub-0.01-per-MCS rates (no observable integer
population growth from the chemokine/APC signal within a short run, only
from the baseline term at a boosted eta); and the nearby-killing term,
grown via natural recruitment alone over 300 MCS, remains orders of
magnitude too weak to fire. These are FIELD-vs-ODE UNIT-SCALE
CALIBRATION gaps — the same root-cause category as Increment 7's
chemotaxis-engine-scale correction — deliberately reported as open,
NOT tuned away, and deferred to Increment 9.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_cpm_studies.composites.influenza.systemic_ode` | 0 | chemotaxis_v_macro=5000.0, chemotaxis_v_nk=500000.0, chemotaxis_v_cd8=1000000.0, seed=17 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.influenza.systemic_ode`** — `spec_pbg_cpm_studies_composites_influenza_systemic_ode` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.influenza.systemic_ode` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: global-coupling ===
STUDY = 'global-coupling'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Spatial state — global immune scene (animated)**


In [ ]:
# Spatial state — global immune scene (animated)
show_viz(_render_one('local:InfluenzaSpatialGlobalCoupling', {}, RUNS_DB, STUDY_YAML))

**Hybrid global ODE coupling**


In [ ]:
# Hybrid global ODE coupling
show_viz(_render_one('local:InfluenzaGlobalCoupling', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| GlobalODE integrates the 10 systemic species stably over a run | kind=ode_states_finite_and_stable condition=global-coupling-ode-stability stat=final | op eq value True |
| Dynamic sig_1 differs from and replaces the static stub | kind=sig_1_dynamic_monotone_and_differs_from_stub condition=dynamic-sig1-vs-stub stat=final | op eq value True |
| CD8+ recruitment inflow is APC-driven with no homeostatic baseline | kind=cd8_recruitment_is_apc_driven_asymmetry_preserved condition=cd8-no-baseline-recruitment stat=final | op eq value True |
| Nearby-population killing clears infected cells given a sufficient nearby population | kind=n_infected_final_lower_with_nearby_surrogate_present condition=nearby-killing-on-vs-off stat=final | op lt value n_infected_final[K_nb=E_nb=0 control] |
| DH-input fix -- dead-from-infected cells do not inflate the dead-from-healthy ODE input | kind=dh_input_zero_regardless_of_killing condition=dh-input-fix stat=final | op eq value True |
| global_coupling_figure renders a non-vacuous 2-panel mechanism figure | kind=figure_axes_and_data_match_driver_output condition=global-coupling-figure-non-vacuous stat=final | op eq value True |


## Study: Reproduction — Fig 3B (time series vs ODE) (`repro-fig3b`)

**Question.** Does `run.repro_fig3b` (Task 9.2) correctly assemble a seeded ensemble of
`run.run_full_model` (Task 9.1) over the Sego-2022 Fig-3B scenario, map
its output onto `targets/fig3b.json`'s exact observable keys, and evaluate
the ensemble mean against the digitized Fig-3B acceptance bands
(`bands.evaluate_study`) -- and, honestly, does a REDUCED-scale ensemble
(the only scale a fast CI test can run) land in-band, or does it mostly
expose the reduced population/duration's mismatch with the paper-scale
bands rather than any mechanism defect?

**Objective.** Implement `run.repro_fig3b(*, replicas=3, cells_per_side=35, steps=240,
seed0=0) -> dict` (Task 9.2): loop `replicas` seeds through
`run.run_full_model(cells_per_side=cells_per_side, steps=steps,
seed=seed0+i, init_infection_frac=0.05)`; map each run's
`counts`/`fields`/`ode` sections onto `targets/fig3b.json`'s exact
observable keys (`_FIG3B_OBSERVABLE_MAP`); ensemble-mean via
`bands.aggregate_replicas`; evaluate via `bands.evaluate_study(ensemble,
"fig3b")`. Add `tests/test_influenza_full_model.py::
test_repro_fig3b_runs_and_evaluates_bands` (reduced: `replicas=2,
cells_per_side=15, steps=20`) asserting the ensemble/band_eval dict is
well-formed and the uninfected series is non-vacuous (declines under
infection) -- NOT that the bands pass, since reduced scale is not
expected to. Run that same reduced config directly (not the pytest
assertions alone) to report the actual per-observable band_eval numbers
in this study honestly.

**Hypothesis.** `run.repro_fig3b` correctly wires the ensemble-and-band-evaluation
machinery: it runs `replicas` seeded `run_full_model` calls, maps every
mapped observable's series onto the fig3b target's keys, ensemble-means
them, and produces a well-formed `band_eval` dict. At REDUCED scale
(small `cells_per_side`, few `replicas`, short `steps` -- all needed to
keep the added pytest test fast), the ensemble is NOT expected to land
in-band against bands digitized from a 1225-cell, 50-replica, ~3.5-day
paper-scale ensemble; any near-miss or in-band result at reduced scale
should be reported as what it is (a coincidence of a widened `soft` band
and/or an early-timepoint value that happens to still be small), not
oversold as reproduction evidence.

**Purpose.** full_model_ensemble_vs_fig3b_acceptance_bands

**Claim.** `run.repro_fig3b` is implemented and wired correctly: it runs a seeded
ensemble of `run_full_model` over the Fig-3B scenario, maps every
observable onto `targets/fig3b.json`'s exact keys, ensemble-means via
`bands.aggregate_replicas`, and evaluates via `bands.evaluate_study`,
producing a well-formed `{"ensemble":..., "band_eval":{"passed": bool,
...}, "replicas":..., "cells_per_side":..., "steps":...}` dict (verified
by `tests/test_influenza_full_model.py::
test_repro_fig3b_runs_and_evaluates_bands`). THIS STUDY DOES NOT CLAIM
FIG-3B IS REPRODUCED. At the reduced scale this task's fast test uses
(`replicas=2, cells_per_side=15, steps=20, seed0=0`), `band_eval['passed']`
is False and 0 of the 12 fig3b observables are in-band at every
checkpoint (per-observable n_in/8 and worst_miss reported in `report.
key_metrics` above) -- expected given the ~5.4x smaller epithelial
population and the ~0.09-simulated-day run duration vs. the target's
0.0-3.5-day checkpoint window, not a mechanism defect. The paper-scale
50-replica, 35x35-cell, ~3.5-day ensemble (Mac-mini Phase-B) is required
before any `reproduced` verdict for Fig 3B, and `conclusion_verdicts.
biological_validation` stays PENDING here.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_cpm_studies.composites.influenza.full_model` | 20 | replicas=2, cells_per_side=15, seed0=0 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.influenza.full_model`** — `spec_pbg_cpm_studies_composites_influenza_full_model` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.influenza.full_model` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: repro-fig3b ===
STUDY = 'repro-fig3b'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Spatial state — full model (animated)**


In [ ]:
# Spatial state — full model (animated)
show_viz(_render_one('local:InfluenzaSpatialReproFig3B', {}, RUNS_DB, STUDY_YAML))

**Fig-3B reproduction — ensemble vs acceptance band**


In [ ]:
# Fig-3B reproduction — ensemble vs acceptance band
show_viz(_render_one('local:InfluenzaReproFig3B', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| repro_fig3b runs and returns a well-formed ensemble/band evaluation | kind=repro_fig3b_wellformed_and_uninfected_nonincreasing condition=repro-fig3b-reduced-scale stat=final | op eq value True |


## Study: Reproduction — Fig 5 (viral-load sweep) (`repro-fig5-viral-load`)

**Question.** Does `run.repro_fig5` (Task 9.3) correctly sweep `run.run_full_model`
(Task 9.1) over Sego-2022 Fig-5's initial-viral-load scenarios, evaluate
EACH load against ONLY that load's own fig5.json band subset (never the
unfiltered, multi-scenario list), and reproduce the paper's central
monotone, threshold-like dose-response direction -- and, honestly, does a
REDUCED-scale sweep (the only scale a fast CI test can run) show that
direction even though it cannot match the digitized bands' absolute
magnitudes at paper scale?

**Objective.** Implement `run.repro_fig5(*, loads=(1,10,100,1000,10000), replicas=3,
cells_per_side=35, steps=240, seed0=0) -> dict` (Task 9.3): for each
`load`, loop `replicas` seeds through `run.run_full_model(
cells_per_side=cells_per_side, steps=steps, seed=seed0+load_idx*replicas+r,
init_viral_load=load, init_infection_frac=None)`; map each run's
`counts`/`fields`/`ode` sections onto `targets/fig5.json`'s exact 4
observable keys (`_FIG5_OBSERVABLE_MAP`); ensemble-mean via
`bands.aggregate_replicas`; filter that load's own band subset from
fig5.json's multi-scenario observable lists (`_fig5_target_subset`,
raising `ValueError` rather than silently mixing scenarios if a load has
no matching entries); evaluate via `_evaluate_fig5_subset`. Add
`tests/test_influenza_full_model.py::test_repro_fig5_viral_load_sweep`
(reduced: `loads=(1,10000), replicas=1, cells_per_side=12, steps=15`)
asserting the by_load dict has exactly the tested loads as keys, the
monotone dose-response holds (`uninfected_final_frac(10000) <=
uninfected_final_frac(1) + 1e-9`), and `band_eval` is present -- NOT
that the bands pass, since reduced scale is not expected to. Run that
same reduced config directly (not the pytest assertions alone) to report
the actual per-load, per-observable band_eval numbers in this study
honestly.

**Hypothesis.** `run.repro_fig5` correctly wires the per-load ensemble-and-scenario-
filtered-band-evaluation machinery: for each tested load it runs
`replicas` seeded `run_full_model` calls with a uniform virus-field IC,
maps every observable onto fig5's keys, ensemble-means them, filters
fig5.json's observable lists down to that load's own scenario subset
(the guard), and produces a well-formed per-load `band_eval`. At REDUCED
scale (few loads/replicas, a small population, a short duration -- all
needed to keep the added pytest test fast), the ensemble is NOT expected
to land in-band against bands digitized from a 10000-cell, 50-replica,
15-day paper-scale ensemble, but the MONOTONE dose-response direction
(higher initial viral load -> not-more surviving uninfected fraction)
should still hold, since it is a qualitative mechanism check independent
of population/time scale; any near-miss or in-band result at reduced
scale should be reported as what it is (a coincidence of a wide,
paper-reported-high-spread band and/or a near-instant collapse that
happens to land inside it), not oversold as reproduction evidence.

**Purpose.** full_model_viral_load_sweep_vs_fig5_acceptance_bands

**Claim.** `run.repro_fig5` is implemented and wired correctly: it sweeps a seeded
ensemble of `run_full_model` over Fig-5's viral-load scenarios, maps
every observable onto `targets/fig5.json`'s exact keys, ensemble-means
per load via `bands.aggregate_replicas`, filters each load's own
scenario-tagged band subset (`_fig5_target_subset`, verified to raise
`ValueError` rather than silently mixing scenarios when a load has no
tagged entries), and evaluates via `_evaluate_fig5_subset`, producing a
well-formed `{"by_load": {load: {...}}, "lethal_threshold": ...,
"band_eval": {...}, "loads":..., "replicas":..., "cells_per_side":...,
"steps":...}` dict (verified by `tests/test_influenza_full_model.py::
test_repro_fig5_viral_load_sweep`). THIS STUDY DOES NOT CLAIM FIG-5 IS
REPRODUCED. At the reduced scale this task's fast test uses
(`loads=(1,10000), replicas=1, cells_per_side=12, steps=15, seed0=0`),
per-load `band_eval["passed"]` is False for both loads (per-observable
n_in/6 and worst_miss reported in `report.key_metrics` above) -- expected
given the ~69x smaller epithelial population and the ~0.068-simulated-day
run duration vs. the target's 0.0-15-day checkpoint window, not a
mechanism defect. The MONOTONE dose-response direction this reduced
test's own assertion checks (`uninfected_final_frac(load=10000)=0.0 <=
uninfected_final_frac(load=1)=0.611`) holds. The paper-scale 50-replica,
all-5-load, 35x35-cell, ~15-day ensemble (Mac-mini Phase-B) is required
before any `reproduced` verdict for Fig 5, and `conclusion_verdicts.
biological_validation` stays PENDING here.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_cpm_studies.composites.influenza.full_model` | 15 | loads=[1, 10000], replicas=1, cells_per_side=12, seed0=0 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.influenza.full_model`** — `spec_pbg_cpm_studies_composites_influenza_full_model` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.influenza.full_model` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: repro-fig5-viral-load ===
STUDY = 'repro-fig5-viral-load'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Spatial state — full model (animated)**


In [ ]:
# Spatial state — full model (animated)
show_viz(_render_one('local:InfluenzaSpatialReproFig5', {}, RUNS_DB, STUDY_YAML))

**Fig-5 reproduction — dose-response vs acceptance band**


In [ ]:
# Fig-5 reproduction — dose-response vs acceptance band
show_viz(_render_one('local:InfluenzaReproFig5', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| repro_fig5 sweeps viral loads and returns a well-formed by_load/band_eval dict with the monotone dose-response | kind=repro_fig5_wellformed_and_monotone_dose_response condition=repro-fig5-reduced-scale stat=final | op eq value True |


## Study: Reproduction — Fig 7 (infection-fraction sweep) (`repro-fig7-infection-fraction`)

**Question.** Does `run.repro_fig7` (Task 9.4) correctly sweep `run.run_full_model`
(Task 9.1) over Sego-2022 Fig-7's initial-infection-fraction scenarios,
evaluate EACH fraction against ONLY that fraction's own fig7.json band
subset (never the unfiltered, multi-scenario list, via the shared
scenario-grouping guard generalized from `repro_fig5`), and reproduce the
paper's central threshold-like severity-response direction -- and,
honestly, does a REDUCED-scale sweep (the only scale a fast CI test can
run) show that direction even though it cannot match the digitized bands'
absolute magnitudes at paper scale, and does it surface any NEW
reduced-scale artifacts beyond what `repro_fig5`/`repro_fig3b` already
documented?

**Objective.** Implement `run.repro_fig7(*, fracs=(0.001,0.005,0.01,0.05), replicas=3,
cells_per_side=35, steps=240, seed0=0) -> dict` (Task 9.4): for each
`frac`, loop `replicas` seeds through `run.run_full_model(
cells_per_side=cells_per_side, steps=steps, seed=seed0+frac_idx*replicas+r,
init_infection_frac=frac, init_viral_load=None)`; map each run's
`counts`/`fields`/`ode` sections onto `targets/fig7.json`'s exact 4
observable keys (reusing `_FIG5_OBSERVABLE_MAP`); ensemble-mean via
`bands.aggregate_replicas`; filter that fraction's own band subset from
fig7.json's multi-scenario observable lists (`_fig7_target_subset`,
raising `ValueError` rather than silently mixing scenarios if a fraction
has no matching entries); evaluate via `_evaluate_fig7_subset`. Add
`tests/test_influenza_full_model.py::
test_repro_fig7_infection_fraction_sweep` (reduced:
`fracs=(0.001,0.05), replicas=1, cells_per_side=12, steps=15`) asserting
the by_frac dict has exactly the tested fractions as keys, the monotone
dose-response holds (`uninfected_final_frac(0.05) <=
uninfected_final_frac(0.001) + 1e-9`), and `band_eval` is present -- NOT
that the bands pass. ALSO add
`test_fig7_target_subset_scenario_grouping_guard`, a fast unit test
directly exercising `_fig7_target_subset`'s filter (returns only the
matching-tag entries) and its raise (unmatched fraction), closing the
Task-9.3-review-flagged gap that the guard was previously verified only
interactively. Run the reduced config directly (not the pytest assertions
alone) to report the actual per-fraction, per-observable band_eval numbers
in this study honestly.

**Hypothesis.** `run.repro_fig7` correctly wires the per-fraction ensemble-and-scenario-
filtered-band-evaluation machinery, reusing `repro_fig5`'s shared
`_scenario_target_subset` guard rather than duplicating it: for each
tested fraction it runs `replicas` seeded `run_full_model` calls with
`round(frac*tot_cell)` pre-infected cells, maps every observable onto
fig7's keys, ensemble-means them, filters fig7.json's observable lists
down to that fraction's own scenario subset (the guard), and produces a
well-formed per-fraction `band_eval`. At REDUCED scale (few fractions/
replicas, a small population, a short duration -- all needed to keep the
added pytest test fast), the ensemble is NOT expected to land in-band
against bands digitized from a 10000-cell, 20-replica, 15-day paper-scale
ensemble, but the MONOTONE dose-response direction (higher initial
infection fraction -> not-more surviving uninfected fraction) should
still hold. A risk specific to `init_infection_frac` (unlike fig5's
`init_viral_load`, which has no analogous rounding step): the smallest
fraction may round to zero seeded cells at a small `cells_per_side`,
producing a degenerate no-infection run that should be reported as what
it is, not oversold as reproduction evidence.

**Purpose.** full_model_infection_fraction_sweep_vs_fig7_acceptance_bands

**Claim.** `run.repro_fig7` is implemented and wired correctly: it sweeps a seeded
ensemble of `run_full_model` over Fig-7's initial-infection-fraction
scenarios, maps every observable onto `targets/fig7.json`'s exact keys
(reusing fig5's `_FIG5_OBSERVABLE_MAP` since both targets share the same 4
keys), ensemble-means per fraction via `bands.aggregate_replicas`, filters
each fraction's own scenario-tagged band subset (`_fig7_target_subset`,
built on the shared `_scenario_target_subset` this task generalized from
`repro_fig5`'s guard, verified to raise `ValueError` rather than silently
mixing scenarios when a fraction has no tagged entries, AND verified to
filter correctly for the happy path -- both now covered by a dedicated
pytest unit test), and evaluates via `_evaluate_fig7_subset`, producing a
well-formed `{"by_frac": {frac: {...}}, "lethal_threshold": ...,
"band_eval": {...}, "fracs":..., "replicas":..., "cells_per_side":...,
"steps":...}` dict (verified by `tests/test_influenza_full_model.py::
test_repro_fig7_infection_fraction_sweep`). THIS STUDY DOES NOT CLAIM
FIG-7 IS REPRODUCED. At the reduced scale this task's fast test uses
(`fracs=(0.001,0.05), replicas=1, cells_per_side=12, steps=15, seed0=0`),
per-fraction `band_eval["passed"]` is False for both fractions
(per-observable n_in/6 and worst_miss reported in `report.key_metrics`
above) -- expected given the ~69x smaller epithelial population and the
~0.068-simulated-day run duration vs. the target's 0.0-15-day checkpoint
window, not a mechanism defect. The MONOTONE dose-response direction this
reduced test's own assertion checks
(`uninfected_final_frac(frac=0.05)=0.9444 <=
uninfected_final_frac(frac=0.001)=1.0`) holds, BUT frac=0.001's value is a
degenerate no-infection control at this reduced population
(`round(0.001*144)=0` pre-infected cells seeded), reported here honestly
rather than presented as a real reproduction of fig7.json's frac=0.001
trajectory. The paper-scale 20-replica, all-4-fraction, 35x35-cell,
~15-day ensemble (Mac-mini Phase-B) is required before any `reproduced`
verdict for Fig 7, and `conclusion_verdicts.biological_validation` stays
PENDING here.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_cpm_studies.composites.influenza.full_model` | 15 | fracs=[0.001, 0.05], replicas=1, cells_per_side=12, seed0=0 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.influenza.full_model`** — `spec_pbg_cpm_studies_composites_influenza_full_model` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.influenza.full_model` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: repro-fig7-infection-fraction ===
STUDY = 'repro-fig7-infection-fraction'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Spatial state — full model (animated)**


In [ ]:
# Spatial state — full model (animated)
show_viz(_render_one('local:InfluenzaSpatialReproFig7', {}, RUNS_DB, STUDY_YAML))

**Fig-7 reproduction — dose-response vs acceptance band**


In [ ]:
# Fig-7 reproduction — dose-response vs acceptance band
show_viz(_render_one('local:InfluenzaReproFig7', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| repro_fig7 sweeps infection fractions and returns a well-formed by_frac/band_eval dict with the monotone dose-response | kind=repro_fig7_wellformed_and_monotone_dose_response condition=repro-fig7-reduced-scale stat=final | op eq value True |
| Scenario-grouping guard filters fig7's tagged observables correctly and raises loudly on an unmatched fraction | kind=fig7_scenario_grouping_guard_filter_and_raise condition=repro-fig7-reduced-scale stat=final | op eq value True |


## Open decisions
- Should any of the 8 source-vs-paper discrepancies (sego2022-parameters.md §7) be resolved toward the paper's stated values instead of the source-literal ones before Increment 1 locks in params.yaml as ground truth?
